# Beginner Tutorial 5: Cloud Computing Fundamentals

## What You Will Learn

- What cloud computing is (renting computers over the internet)
- Core cloud services: compute, storage, networking
- Object storage (S3/GCS) vs. filesystems
- Containers and why they matter for cloud
- How to configure cloud targets in Scalable
- Cost estimation before running

## Prerequisites

- Completed notebooks 01–02
- `pip install scalable[cloud]` (for code examples)
- NO cloud account required (conceptual + configuration examples)

In [ ]:
import os
import tempfile

project_dir = tempfile.mkdtemp(prefix="scalable-beginner-05-")
os.chdir(project_dir)
print(f"Working directory: {project_dir}")

## 💡 Key Concept: What is Cloud Computing?

**Cloud computing** = renting computers, storage, and networking over the internet.
Pay only for what you use.

**Before cloud:** Buy servers, install them, maintain them, pay whether used or idle.

**With cloud:** Request "10 machines for 2 hours" → they appear in seconds → stop paying when done.

## 💡 Key Concept: Object Storage (S3/GCS)

**Object storage** = cloud service for storing files in "buckets":
```
s3://my-bucket/scalable-runs/results.json
│    │         │              │
│    bucket    prefix         object key
└── protocol
```

Why not a regular filesystem?
- Scales to petabytes
- 99.999999999% durability (11 nines!)
- Accessible from anywhere
- Very cheap ($0.023/GB/month)

## 💡 Key Concept: Containers

A **container** packages code + all dependencies into a portable unit.
Solves the "works on my machine" problem.

**Analogy:** Like a shipping container — the crane doesn't care what's inside,
it just knows how to move the standard container.

In [ ]:
# Example: Cloud target configuration in a manifest
cloud_manifest = """\
version: 1
project:
  name: energy-model
  default_storage: s3://my-bucket/scalable-runs/

targets:
  local:
    provider: local
    max_workers: 4
    processes: false
    containers: none

  aws:
    provider: aws
    region: us-east-1
    cluster_type: fargate
    worker_cpu: 4096       # 4 vCPUs (Fargate units: 1024 = 1 vCPU)
    worker_mem: 16384      # 16 GB (in MB)
    image: 123456789.dkr.ecr.us-east-1.amazonaws.com/energy-model:latest
    adaptive:
      minimum: 1
      maximum: 10

components:
  analysis:
    cpus: 4
    memory: 16G

tasks:
  run_analysis:
    component: analysis
"""

with open("scalable.yaml", "w") as f:
    f.write(cloud_manifest)

print("Cloud manifest written.")
print("\nKey cloud settings explained:")
print("  region: us-east-1 → which data center")
print("  cluster_type: fargate → serverless containers (no servers to manage)")
print("  image: ... → container with your code & dependencies")
print("  adaptive: min=1, max=10 → auto-scale based on demand")

## 💡 Key Concept: Cloud Cost Model

Cloud charges for what you use:

| Resource | Pricing | Example |
|----------|---------|--------|
| Compute | Per-second while running | $0.04/vCPU-hour |
| Storage | Per-GB-month | $0.023/GB-month |
| Network | Per-GB transferred out | $0.09/GB |

**Example run cost:**
```
10 workers × 4 vCPU × 2 hours × $0.04/vCPU-hour = $3.20
10 workers × 16GB × 2 hours × $0.004/GB-hour     = $1.28
Output: 50GB × $0.023/GB-month                    = $1.15/month
─────────────────────────────────────────────────────
Total: ~$5.63 (one-time) + $1.15/month (storage)
```

Scalable's `--dry-run` flag estimates costs BEFORE running!

In [ ]:
from scalable.manifest.parser import load_manifest
from scalable.manifest.validate import validate_manifest

# Validate the cloud manifest
manifest = load_manifest("./scalable.yaml")
report = validate_manifest(manifest)

if not report.ok:
    for issue in report.errors:
        print(f"  ✗ {issue.path}: {issue.message}")
else:
    print("✓ Cloud manifest is valid")
    print(f"\nTargets available: {list(manifest.targets.keys())}")
    print(f"\n💡 Same manifest works locally AND in the cloud!")
    print(f"   Development: scalable run --target local")
    print(f"   Production:  scalable run --target aws")

## 🤔 Think About It

Notice how the **Python code doesn't change** between local and cloud.
Only the `--target` flag changes. This is the power of:
- Declarative manifests (configuration, not code)
- Provider abstraction (same API, different backend)

## 📖 Vocabulary Summary

| Term | Definition |
|------|------------|
| Cloud Computing | Renting resources over internet, pay-per-use |
| Object Storage (S3/GCS) | Cloud file storage in buckets |
| Container | Packaged code + dependencies, runs anywhere |
| Container Registry | Storage for container images (ECR, GCR) |
| IAM | Identity & Access Management (permissions) |
| VPC | Virtual Private Cloud (isolated network) |
| Fargate | AWS serverless container service |
| Region | Geographic location of data center |
| Spot Instance | Cheap compute that can be interrupted |
| Artifact | Workflow output stored for persistence |

In [ ]:
# Cleanup
import shutil
os.chdir("/tmp")
shutil.rmtree(project_dir, ignore_errors=True)
print(f"Cleaned up {project_dir}")